# Capítulo 00: Un RAG de juguete

En este notebook construimos el RAG más simple posible, **sin librerías ni API keys**,
para ver las tres piezas en acción:

1. **Retrieval:** buscar la información relevante.
2. **Augmentation:** meterla en el prompt.
3. **Generation:** pedirle al LLM que responda *(este paso lo haremos de verdad en el capítulo 07)*.

## Nuestra "base de conocimiento"

En un sistema real serían miles de documentos. Para empezar, nos bastan cinco frases
sobre NovaTech, la empresa ficticia del curso.

In [1]:
knowledge_base = [
    "En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.",
    "Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.",
    "La dieta diaria para viajes de trabajo en España es de 45 €.",
    "Si pierdes el portátil o sufres un robo, avisa a la guardia de IT en el 900 000 111.",
    "El plan Starter de NovaDesk cuesta 15 € por agente y mes.",
]

## Paso 1: Retrieval (buscar)

La forma más ingenua de buscar: contar cuántas palabras comparte la pregunta con cada
documento. Antes quitamos las *stopwords*, palabras muy frecuentes como "el" o "de"
que no aportan significado.

In [2]:
import re

STOPWORDS = set(
    "a al con cuál cuántos de del el en es la las lo los me mi para por qué se si tengo un una y".split()
)


def tokenize(text: str) -> set[str]:
    words = re.findall(r"\w+", text.lower())
    return {word for word in words if word not in STOPWORDS}


def search(question: str, documents: list[str], k: int = 2) -> list[str]:
    question_words = tokenize(question)
    scored = [(len(question_words & tokenize(doc)), doc) for doc in documents]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [doc for score, doc in scored[:k] if score > 0]


question = "¿Cuántos días de vacaciones tengo?"
print("Palabras clave:", tokenize(question))
for doc in search(question, knowledge_base):
    print("-", doc)

Palabras clave: {'días', 'vacaciones'}
- En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.
- Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.


## Paso 2: Augmentation (construir el prompt)

Ahora juntamos la pregunta y los fragmentos encontrados en un único prompt. Fíjate en
las instrucciones: le pedimos al modelo que **use solo el contexto** y que admita
cuando no sabe algo. Esto reduce mucho las alucinaciones.

In [3]:
def build_prompt(question: str, context: list[str]) -> str:
    context_text = "\n".join(f"- {fragment}" for fragment in context) or "(sin contexto)"
    return f"""Eres el asistente interno de NovaTech.
Responde a la pregunta usando SOLO la información del contexto.
Si el contexto no contiene la respuesta, di "No tengo esa información".

Contexto:
{context_text}

Pregunta: {question}
Respuesta:"""


print(build_prompt(question, search(question, knowledge_base)))

Eres el asistente interno de NovaTech.
Responde a la pregunta usando SOLO la información del contexto.
Si el contexto no contiene la respuesta, di "No tengo esa información".

Contexto:
- En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.
- Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.

Pregunta: ¿Cuántos días de vacaciones tengo?
Respuesta:


## Paso 3: Generation (responder)

Este prompt es lo que se enviaría a un LLM. Con el contexto delante, el modelo puede
responder algo como:

> Tienes 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.
> Recuerda solicitarlas en NovaHR con al menos 15 días de antelación.

En el capítulo 07 conectaremos un LLM de verdad. De momento, lo importante es ver que
**el modelo ya no necesita "saber" nada sobre NovaTech**: la respuesta está en el prompt.

## ¿Y si la respuesta no está en los documentos?

La búsqueda no encuentra nada, el contexto queda vacío y nuestras instrucciones le
dicen al modelo que responda "No tengo esa información" en vez de inventar.

In [4]:
question = "¿Cuál es el salario de la CEO?"
print(build_prompt(question, search(question, knowledge_base)))

Eres el asistente interno de NovaTech.
Responde a la pregunta usando SOLO la información del contexto.
Si el contexto no contiene la respuesta, di "No tengo esa información".

Contexto:
(sin contexto)

Pregunta: ¿Cuál es el salario de la CEO?
Respuesta:


## Las limitaciones de nuestro RAG de juguete

Prueba esta pregunta. La respuesta **sí** está en la base de conocimiento (la frase
del portátil), pero la búsqueda no la encuentra:

In [5]:
question = "¿Qué pasa si me roban el ordenador?"
print("Palabras clave:", tokenize(question))
print("Resultados:", search(question, knowledge_base))

Palabras clave: {'pasa', 'ordenador', 'roban'}
Resultados: []


"roban" no es igual a "robo" y "ordenador" no es igual a "portátil". Nuestra búsqueda
solo entiende de palabras idénticas, no de **significado**.

Estos son los problemas que resolveremos durante el curso:

| Problema | Capítulo |
|----------|----------|
| Los documentos reales son largos y hay que trocearlos | 02: Chunking |
| Todas las palabras valen lo mismo ("vacaciones" debería pesar más que "año") | 03: BM25 |
| No entiende sinónimos ni significado | 04: Embeddings |
| Buscar en miles de documentos de forma eficiente | 05: Vector store |
| El LLM no responde de verdad | 07: Generación |

➡️ Siguiente: **Capítulo 01: Cargar documentos.**